In [1]:
import requests
import json
from astropy.time import Time


In [3]:
FRITZ_TOKEN = '5b0de79b-ad37-4c8f-a963-8ab59e9320b2'
LCO_TOKEN ='d9b9d2faf4803e4bfd8efe803a2a943e629ca8fc'
requestpath  = "https://observe.lco.global/api/requestgroups/"


# example_path = './example.json'

# with open(example_path, "r") as f:
#         data = json.load(f)

    
# response = requests.post(
#                 requestpath,
#                 headers={"Authorization": f"Token {LCO_TOKEN}"},
#                 json=data,  # Make sure you use json!
#             )



# if response.status_code == 400:
#     print('Failed (sending to queue)')
#     # print(response.text)
#     # print(ra/15)
#     # lco_failed.append(variables['target']['id'])
# if response.status_code == 200 or response.status_code == 201:
#     print('Successfully sent')
    
        
    

In [38]:
# add filter
import copy
import glob
import numpy as np
from datetime import datetime, timedelta
import os
import re

def get_filters(filename):
    base = os.path.splitext(filename)[0]
    
    if '_' not in base:
        return 'grz' 
        
    return base.split('_')[-1]

date = 'May1'
files = np.sort(glob.glob(f'./{date}/*.txt'))

ids,failed = [],[]

for i in range(len(files)):
    targets = np.genfromtxt(files[i])
    filts = get_filters(files[i])

    for ii in range(len(targets)):
    
        example_path = './example.json'
    
        with open(example_path, "r") as f:
            data = json.load(f)
    
        # name, ra, dec
    
        target = data['requests'][0]['configurations'][0]['target']
        target['ra'] = targets[ii][0]
        target['dec'] = targets[ii][1]
        
        
        # filters
        
        instrument_configs = data['requests'][0]['configurations'][0]['instrument_configs']
        instrument_configs[0]['optical_elements']['filter'] = filts[0]
        original_config = instrument_configs[0]
        
        if len(filts) > 1:
            for iii in range(1,len(filts)):
                filt_config = copy.deepcopy(original_config)
                filt_config['optical_elements']['filter'] = filts[iii]
                instrument_configs.append(filt_config)
        
        # time window
        
        t = re.search(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', files[i]).group()
        data['name'] = f'pointing_{t}_{filts}_{ii}'
        target['name'] = f'pointing_{t}_{filts}_{ii}'
        window_original = data['requests'][0]['windows'][0]
        t_obj = datetime.strptime(t, "%Y-%m-%d %H:%M:%S")
        window_original['start'] = (t_obj - timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M:%S")
        window_original['end'] = (t_obj + timedelta(minutes=300)).strftime("%Y-%m-%d %H:%M:%S")
    
        # priority
        data['ipp_value'] = 0.9
        
        # time critical
        # data['observation_type'] = "TIME_CRITICAL"
        
        # posting
        response = requests.post(
                    requestpath,
                    headers={"Authorization": f"Token {LCO_TOKEN}"},
                    json=data,  # Make sure you use json!
                )
    
    
    
        if response.status_code == 400:
            print('Failed (sending to queue)')
            failed.append([data['name'],response.text])
        if response.status_code == 200 or response.status_code == 201:
            print(data['name'],window_original['start'],window_original['end'],filts, response.json()['id'])
            ids.append(response.json()['id'])
# data
np.save(f'./{date}/ids.npy',ids)
np.save(f'./{date}/failed.npy',failed)

pointing_2026-05-01 23:00:00_grz_0 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502358
pointing_2026-05-01 23:00:00_grz_1 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502359
pointing_2026-05-01 23:00:00_grz_2 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502360
pointing_2026-05-01 23:00:00_grz_3 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502362
pointing_2026-05-01 23:00:00_grz_4 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502363
pointing_2026-05-01 23:00:00_grz_5 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502364
pointing_2026-05-01 23:00:00_grz_6 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502366
pointing_2026-05-01 23:00:00_grz_7 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502367
pointing_2026-05-01 23:00:00_grz_8 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502368
pointing_2026-05-01 23:00:00_grz_9 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502370
pointing_2026-05-01 23:00:00_grz_10 2026-05-01 22:30:00 2026-05-02 04:00:00 grz 2502371
pointing_2026-05-01 23:00:00_grz_11 2026-0

In [ ]:
for idlco in ids:
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    print(idlco,response.status_code)


In [ ]:
pointing_2026-04-21 23:45:00_0 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_1 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_2 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_3 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_4 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_5 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_6 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_7 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_8 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_9 2026-04-21 23:45:00 2026-04-22 00:15:00 grz
pointing_2026-04-21 23:45:00_0 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_1 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_2 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_3 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_4 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_5 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_6 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_7 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_8 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_9 2026-04-22 00:15:00 2026-04-22 00:45:00 grz
pointing_2026-04-22 00:15:00_0 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_1 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_2 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_3 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_4 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_5 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_6 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_7 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_8 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_9 2026-04-22 00:45:00 2026-04-22 01:15:00 grz TIME_CRITICAL
pointing_2026-04-22 00:45:00_0 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_1 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_2 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_3 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_4 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_5 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_6 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_7 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_8 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 01:15:00_9 2026-04-22 01:15:00 2026-04-22 01:45:00 grz TIME_CRITICAL
pointing_2026-04-22 02:15:00_9 2026-04-22 01:45:00 2026-04-22 03:45:00 grz 10_targets
pointing_2026-04-22 02:15:00_9 2026-04-22 02:15:00 2026-04-22 04:15:00 grz 10_targets
pointing_2026-04-22 01:45:00_0 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_1 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_2 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_3 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_4 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_5 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_6 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_7 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_8 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_9 2026-04-22 02:45:00 2026-04-22 03:15:00 grz 
pointing_2026-04-22 02:45:00_0 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_1 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_2 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_3 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_4 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_5 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_6 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_7 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_8 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL
pointing_2026-04-22 03:15:00_9 2026-04-22 03:15:00 2026-04-22 03:45:00 grz TIME_CRITICAL



In [110]:
# add filter
import copy
import glob
import numpy as np
from datetime import datetime, timedelta

files = np.sort(glob.glob('./April21/*.txt'))
i=5

targets = np.genfromtxt(files[i])


example_path = './example.json'

with open(example_path, "r") as f:
    data = json.load(f)

# name, ra, dec
data['name'] = f'L3_pointing_{t}_{ii}'
    
    # time window
    
t = files[i][10:-4]
window_original = data['requests'][0]['windows'][0]
t_obj = datetime.strptime(t, "%Y-%m-%d %H:%M:%S")
window_original['start'] = t_obj.strftime("%Y-%m-%d %H:%M:%S")
window_original['end'] = (t_obj + timedelta(minutes=120)).strftime("%Y-%m-%d %H:%M:%S")

for ii in range(3):

    # configuration
    configuration_template = copy.deepcopy(data['requests'][0]['configurations'][0])
    
    target = configuration_template['target']
    target['ra'] = targets[ii][0]
    target['dec'] = targets[ii][1]
    target['name'] = f'L3_pointing_{t}_{ii}'
    
    # filters
    
    instrument_configs = configuration_template['instrument_configs']
    original_config = instrument_configs[0]
    
    g_config = copy.deepcopy(original_config)
    g_config['optical_elements']['filter'] = 'g'
    instrument_configs.append(g_config)
    
    z_config = copy.deepcopy(original_config)
    z_config['optical_elements']['filter'] = 'z'
    instrument_configs.append(z_config)
    
    data['requests'][0]['configurations'].append(configuration_template)

    # time critical
    data['observation_type'] = "TIME_CRITICAL"
    

data['requests'][0]['configurations'].pop(0)

# posting
response = requests.post(
            requestpath,
            headers={"Authorization": f"Token {LCO_TOKEN}"},
            json=data,  # Make sure you use json!
        )

if response.status_code == 400:
    print('Failed (sending to queue)')
if response.status_code == 200 or response.status_code == 201:
    print(data['name'],window_original['start'],window_original['end'],'grz',)

L3_pointing_2026-04-22 02:15:00_4 2026-04-22 02:15:00 2026-04-22 04:15:00 grz


In [111]:
response.text

'{"id":2495571,"requests":[{"id":4168916,"location":{"telescope_class":"4m0"},"configurations":[{"id":14812296,"constraints":{"max_airmass":1.6,"min_lunar_distance":30.0,"max_lunar_phase":1.0,"max_seeing":null,"min_transparency":null,"extra_params":{}},"instrument_configs":[{"exposure_time":40.0,"optical_elements":{"filter":"r"},"mode":"default","exposure_count":1,"rotator_mode":"","extra_params":{"offset_ra":0,"offset_dec":0}},{"exposure_time":40.0,"optical_elements":{"filter":"g"},"mode":"default","exposure_count":1,"rotator_mode":"","extra_params":{"offset_ra":0,"offset_dec":0}},{"exposure_time":40.0,"optical_elements":{"filter":"z"},"mode":"default","exposure_count":1,"rotator_mode":"","extra_params":{"offset_ra":0,"offset_dec":0}}],"acquisition_config":{"mode":"OFF","exposure_time":null,"extra_params":{}},"guiding_config":{"optional":true,"mode":"ON","optical_elements":{},"exposure_time":null,"extra_params":{}},"target":{"type":"ICRS","name":"L3_pointing_2026-04-22 02:15:00_0","ra

In [50]:

files = glob.glob('./April21/*.txt')

i=0
t = files[i][10:-4]
t_obj = datetime.strptime(t, "%Y-%m-%d %H:%M:%S")
window_original['start'] = t
window_original['end'] = (t_obj + timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S")

print(window_original)
# Output: {'start': '2026-04-21 23:45:00', 'end': '2026-04-21 23:55:00'}
# np.genfromtxt(files[0])[0]



{'start': '2026-04-22 04:45:00', 'end': '2026-04-22 04:55:00'}


In [85]:
# idlco = 2495317
for idlco in range(2495357,2495357+10):
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    print(idlco,response.status_code)


2495357 200
2495358 200
2495359 200
2495360 200
2495361 200
2495362 200
2495363 200
2495364 200
2495365 200
2495366 200


In [40]:
def suma(n1,n2): 
    s = n1 + n2
    
    return s




6